#### if you are using databricks make sure to upload the file into volume

In [0]:
path='/Volumes/workspace/default/inputdata/Employee_Dataset_Retail.csv'

In [0]:
df = spark.read.csv(path, header=True, inferSchema=True)
df.createOrReplaceTempView("employee_tbl")

In [0]:
%sql
select * from employee_tbl limit 2

#### salary experiments

In [0]:
%sql
select max(Monthly_Income) from employee_tbl

In [0]:
%sql
-- 2nd highest salary using max
select max(Monthly_Income) from employee_tbl where Monthly_Income not in ( select max(Monthly_Income) from employee_tbl)

In [0]:
%sql
-- 2nd highest salary using limit
select employeeMonthly_Income from employee_tbl where Monthly_Income  in ( select monthly_Income from employee_tbl order by 1 desc limit 2)  order by 1 asc limit 1

In [0]:
%sql
-- 2nd highest salary using dense rank
--Employee_ID
select * from (select Employee_ID,Department,Monthly_Income,dense_rank() over(order by Monthly_Income desc) as rank from employee_tbl) where rank = 2

In [0]:
%sql
-- department wise hightest salary
select * from (select Employee_ID,Department,Monthly_Income,rank() over(partition by Department order by Monthly_Income desc) as rnk from employee_tbl) where rnk =2

In [0]:
%sql
-- department wise hightest salary
select * from (select Employee_ID,Department,Monthly_Income,dense_rank() over(partition by Department order by Monthly_Income desc) as rnk from employee_tbl) where rnk =2

In [0]:
%sql
-- 2nd highest salary using dense rank
--Employee_ID
select * from (select Employee_ID,Department,Monthly_Income,dense_rank() over(order by Monthly_Income desc) as rank from employee_tbl) where rank = 2

In [0]:
%sql
SELECT 
    e1.Monthly_Income
FROM 
    employee_tbl e1
JOIN 
    employee_tbl e2
ON 
    e1.Monthly_Income < e2.Monthly_Income
GROUP BY 
    e1.Monthly_Income
HAVING 
    COUNT(DISTINCT e2.Monthly_Income) = 1;


In [0]:
%sql
select a.Monthly_Income from employee_tbl a join employee_tbl b on a.Monthly_Income < b.Monthly_Income group by a.Monthly_Income having COUNT(DISTINCT b.Monthly_Income) = 1

In [0]:
%sql
select Department,max(calculated_diff) from (select *,Hourly_Rate*160 as calcuated_monthlyrate,(Hourly_Rate*160-Monthly_income) as calculated_diff from employee_tbl) group by all